# Production-Grade MLOps Pipeline for Elderly Health Risk Monitoring
## Interactive Demonstration Notebook

This notebook provides a step-by-step walkthrough of the pipeline:
1. **Dataset Ingestion & Metadata** (UCI NPHA Senior Health Dataset)
2. **Data Validation & Domain Checking**
3. **Preprocessing & Geriatric Feature Engineering**
4. **Champion Random Forest Model & Baseline Comparisons**
5. **Real-time Patient Health Risk Inference**
6. **Operational Telemetry & Data Drift Detection (PSI)**

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set project root
import sys
sys.path.append('..')

from src.utils.config import load_config, resolve_path
from src.data.data_loader import load_raw_data, load_dataset_metadata
from src.preprocessing.preprocessor import load_preprocessor, TARGET_NAMES
from src.training.train import load_champion_model
from src.monitoring.drift_detector import evaluate_data_drift
from src.monitoring.logger import get_monitoring_summary

### 1. Ingested Dataset & Lineage Metadata

In [ ]:
meta = load_dataset_metadata()
print(f"Dataset Name:    {meta['dataset_name']}")
print(f"Source:          {meta['source']}")
print(f"DOI:             {meta['dataset_doi']}")
print(f"Total Records:   {meta['num_instances']}")
print(f"SHA256 Checksum: {meta['sha256_checksum']}")

raw_df = load_raw_data()
raw_df.head()

### 2. Target Class Distribution (Healthcare Utilization / Risk Tiers)

In [ ]:
target_counts = raw_df['Number_of_Doctors_Visited'].value_counts().sort_index()
labels = ['Low Risk (0-1 visits)', 'Moderate Risk (2-3 visits)', 'High Risk (4+ visits)']

plt.figure(figsize=(7, 4))
sns.barplot(x=labels, y=target_counts.values, palette='crest')
plt.title('Elderly Health Risk & Healthcare Utilization Distribution')
plt.ylabel('Patient Count')
plt.xlabel('Risk Tier')
plt.show()

### 3. Model Benchmark Results (Logged from MLflow)

In [ ]:
metrics_path = resolve_path('models/evaluation_metrics.json')
with open(metrics_path, 'r') as f:
    metrics = json.load(f)

results_df = pd.DataFrame([
    {
        'Model': name,
        'Accuracy': f"{data['accuracy']:.4f}",
        'Macro F1': f"{data['f1_macro']:.4f}",
        'Weighted F1': f"{data['f1_weighted']:.4f}"
    }
    for name, data in metrics.items()
])
results_df

### 4. Interactive Health Risk Prediction Demo

In [ ]:
model = load_champion_model()
preprocessor = load_preprocessor()

# Sample frail senior profile: chronic pain, multi-factor sleep issues
frail_patient = pd.DataFrame([{
    'Age': 2,
    'Physical_Health': 4,
    'Mental_Health': 3,
    'Dental_Health': 4,
    'Employment': 4,
    'Stress_Keeps_Patient_from_Sleeping': 1,
    'Medication_Keeps_Patient_from_Sleeping': 1,
    'Pain_Keeps_Patient_from_Sleeping': 1,
    'Bathroom_Needs_Keeps_Patient_from_Sleeping': 1,
    'Uknown_Keeps_Patient_from_Sleeping': 0,
    'Trouble_Sleeping': 2,
    'Prescription_Sleep_Medication': 2,
    'Race': 1,
    'Gender': 2
}])

trans = preprocessor.transform(frail_patient)
pred_class = int(model.predict(trans)[0])
probs = model.predict_proba(trans)[0]

print(f"Predicted Risk Tier: {TARGET_NAMES[pred_class]}")
print(f"Confidence:          {probs[pred_class]:.2%}")
print("Class Probabilities:")
for i, p in enumerate(probs):
    print(f"  - {TARGET_NAMES[i]}: {p:.2%}")

### 5. Production Observability & Data Drift (PSI)

In [ ]:
drift_report = evaluate_data_drift()
print(f"Overall Drift Status:    {drift_report.get('overall_status')}")
print(f"Max Feature PSI:         {drift_report.get('max_psi')}")
print(f"Retrain Recommended:     {drift_report.get('retrain_recommended')}")
print(f"Action Recommendation:   {drift_report.get('recommendation')}")

summary = get_monitoring_summary()
print(f"\nTotal Inferences Logged: {summary['total_predictions']}")
print(f"Average Latency (ms):    {summary['average_latency_ms']} ms")
print(f"Risk Tier Breakdown:     {summary['risk_tier_distribution']}")